# Descàrrega de dades de generació renovable — ENTSO-E
**TFG Arnau Ropero — Grau GCIS, UAB**

Aquest notebook descarrega les dades horàries de generació solar fotovoltaica i eòlica peninsular d'Espanya (2023–2025) via l'API d'ENTSO-E Transparency Platform.

**Font:** ENTSO-E Transparency Platform (`transparency.entsoe.eu`)  
**Sortida:** `generacio_renovable_ES_2023-2025.csv`

## 0. Instal·lació de dependències

In [ ]:
# Executa aquesta cel·la només la primera vegada
!pip install entsoe-py pandas

## 1. Imports i configuració

In [ ]:
import pandas as pd
from entsoe import EntsoePandasClient

# ── CONFIGURACIÓ ──────────────────────────────────────────────────
ENTSOE_API_KEY = "9bfc695e-472d-4a7f-bce1-28fc2d81f552"   
COUNTRY        = "ES"                         # Espanya peninsular
OUTPUT_FILE    = "generacio_renovable_ES_2023-2025.csv"

# Rang temporal — igual que el dataset meteorològic d'Open-Meteo
START = pd.Timestamp("2023-01-01", tz="Europe/Madrid")
END   = pd.Timestamp("2025-06-01", tz="Europe/Madrid")  # exclusiu
# ──────────────────────────────────────────────────────────────────

print(f"Configuració OK")
print(f"  País:    {COUNTRY}")
print(f"  Inici:   {START.date()}")
print(f"  Fi:      {END.date()}")
print(f"  Sortida: {OUTPUT_FILE}")

## 2. Descàrrega de dades

La llibreria `EntsoePandasClient` gestiona automàticament períodes llargs (fa múltiples crides internes any a any). Pot trigar 1–2 minuts.

In [ ]:
client = EntsoePandasClient(api_key=ENTSOE_API_KEY)

print("Descarregant dades de generació real per tecnologia...")
print("(pot trigar 1-2 minuts)")

df_raw = client.query_generation(
    COUNTRY,
    start=START,
    end=END,
    psr_type=None   # totes les tecnologies; filtrarem a la cel·la següent
)

print(f"\nDades rebudes correctament!")
print(f"  Registres: {len(df_raw)}")
print(f"  Tecnologies disponibles ({len(df_raw.columns)}):")
for col in df_raw.columns:
    print(f"    {col}")

## 3. Inspecció del DataFrame raw

In [ ]:
df_raw.head(10)

## 4. Filtratge: solar FV i eòlica

In [ ]:
cols = df_raw.columns.tolist()

def col_str(c):
    return str(c).lower()

# Identifica columnes de solar i eòlica
solar_cols = [c for c in cols if "solar" in col_str(c) or "photovoltaic" in col_str(c)]
wind_cols  = [c for c in cols if "wind" in col_str(c)]

print(f"Columnes solar identificades:  {solar_cols}")
print(f"Columnes eòlica identificades: {wind_cols}")

In [ ]:
# Construeix el DataFrame final
df_out = pd.DataFrame(index=df_raw.index)

if solar_cols:
    df_out["solar_mw"] = df_raw[solar_cols].sum(axis=1)
else:
    print("AVÍS: No s'han trobat columnes de solar FV. Revisa els noms de columna més amunt.")
    df_out["solar_mw"] = float("nan")

if wind_cols:
    df_out["eolica_mw"] = df_raw[wind_cols].sum(axis=1)
else:
    print("AVÍS: No s'han trobat columnes d'eòlica. Revisa els noms de columna més amunt.")
    df_out["eolica_mw"] = float("nan")

# Elimina files amb tot NaN
df_out = df_out.dropna(how="all")
df_out.index.name = "timestamp"
df_out = df_out.reset_index()

print(f"DataFrame final: {len(df_out)} registres")
df_out.head(10)

In [ ]:
# Converteix timestamp a horari (mitjana de cada hora)
df_out["timestamp"] = pd.to_datetime(df_out["timestamp"], utc=True)
df_hourly = df_out.set_index("timestamp").resample("1h").mean()
df_hourly = df_hourly.reset_index()

print(f"Registres originals (15 min): {len(df_out)}")
print(f"Registres agregats (horari):  {len(df_hourly)}")
df_hourly.head(5)

## 5. Estadístiques i validació

In [ ]:
print("=" * 50)
print("ESTADÍSTIQUES")
print("=" * 50)
print(f"Rang temporal: {df_out['timestamp'].min()} → {df_out['timestamp'].max()}")
print(f"Registres totals: {len(df_out)}")
print()
print(f"Solar FV:")
print(f"  Màxim:  {df_out['solar_mw'].max():.0f} MW")
print(f"  Mitjana: {df_out['solar_mw'].mean():.0f} MW")
print(f"  Valors nuls: {df_out['solar_mw'].isna().sum()}")
print()
print(f"Eòlica:")
print(f"  Màxim:  {df_out['eolica_mw'].max():.0f} MW")
print(f"  Mitjana: {df_out['eolica_mw'].mean():.0f} MW")
print(f"  Valors nuls: {df_out['eolica_mw'].isna().sum()}")

In [ ]:
# Comprovació: nombre de registres esperat després d'agregació horària
# 2023-01-01 fins 2025-05-31 = ~21.168 hores
esperats = 21168
reals = len(df_hourly)
diferencia = abs(esperats - reals)

print(f"Registres esperats (aprox): {esperats}")
print(f"Registres obtinguts:        {reals}")
print(f"Diferència:                 {diferencia}")

if diferencia < 200:
    print("\nOK — El nombre de registres és coherent.")
else:
    print("\nAVÍS — Diferència gran, pot haver-hi forats de dades.")

## 6. Guardar CSV

In [ ]:
df_hourly.to_csv(OUTPUT_FILE, index=False)
print(f"Fitxer guardat: {OUTPUT_FILE}")
print("Llest per fusionar amb el dataset meteorològic d'Open-Meteo!")